# 01. Data Understanding and Initial Parsing

## Objective

The goal of this notebook is to build the first analytical dataset for the project from the raw character-level screenplay files.

At this stage, the focus is not yet on modeling or interpretation of characters, but on understanding the structure of the available data and transforming it into a clean tabular format suitable for later analysis.

More specifically, this notebook aims to:

- inspect the structure of the character-level text files
- validate that the raw files follow the expected format
- apply the parsing functions implemented in `src/`
- build an initial dataset with one row per character text entry
- perform a first round of data quality checks

## Why this step matters

All subsequent stages of the project depend on having a reliable base table.  
Before analyzing emotions, building interaction graphs, or studying character evolution, we need to confirm that the character files are consistent and that scene approximations based on `(i, k)` can be used safely.

This notebook therefore serves as the bridge between the raw corpus and the first structured dataset stored in `data/interim/`.

## Expected output

By the end of this notebook, we expect to obtain:

- a validated understanding of the raw character file structure
- a parsed dataset with one row per character text line
- a first set of quality checks to detect inconsistencies or edge cases
- an intermediate dataset ready for preprocessing

---

## 1. Environment setup

In this section, we import the required libraries, define the project root, and load the reusable functions created in `src/`.

This step ensures that the notebook remains clean and focused on analysis, while the parsing logic stays modular and reusable outside the notebook.

In [ ]:
from pathlib import Path
import sys

# Add project root to path to find src module
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
from src.utils.paths import PROJECT_ROOT

# Display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

# Project root is now imported from src.utils.paths
print("Project root:", PROJECT_ROOT)

# Local modules
from src.data_loading import list_character_files
from src.parsing import (
    parse_line,
    parse_character_file,
    build_character_lines_dataset,
)

## 2A. Raw data path diagnosis

The file inventory returned zero character files, which suggests that the dataset path is not being resolved correctly yet.

Before continuing with parsing, we need to verify:

- the current project root being used by the notebook
- whether the expected raw-data directories actually exist
- whether the character files are stored under a different path or extension

This is a required validation step because the rest of the notebook depends on reading the raw character corpus correctly.

In [ ]:
from pathlib import Path

# Use the standardized PROJECT_ROOT
print("Current project root:")
print(PROJECT_ROOT)

print("\nExpected raw paths:")
paths_to_check = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "data" / "raw",
    PROJECT_ROOT / "data" / "raw" / "movie_characters",
    PROJECT_ROOT / "data" / "raw" / "movie_characters" / "texts",
]

for p in paths_to_check:
    print(f"{p} -> exists: {p.exists()}")

## 2B. Directory inspection

After checking the expected paths, we now inspect the available folders directly.

The goal is to understand whether:

- the dataset is stored under a different directory
- the folder names differ from the expected structure
- the project root should be adjusted

In [ ]:
# Inspect top-level project folders
print("Top-level contents of PROJECT_ROOT:\n")
for p in sorted(PROJECT_ROOT.iterdir()):
    print(p.name)

In [ ]:
data_dir = PROJECT_ROOT / "data"

if data_dir.exists():
    print("\nContents of data/:")
    for p in sorted(data_dir.iterdir()):
        print(p.name)

In [ ]:
raw_dir = PROJECT_ROOT / "data" / "raw"

if raw_dir.exists():
    print("\nContents of data/raw/:")
    for p in sorted(raw_dir.iterdir()):
        print(p.name)

In [ ]:
mc_dir = PROJECT_ROOT / "data" / "raw" / "movie_characters"

if mc_dir.exists():
    print("\nContents of data/raw/movie_characters/:")
    for p in sorted(mc_dir.iterdir()):
        print(p.name)

## 2C. Search for candidate files

If the expected folder exists but no files were detected, we search for candidate files recursively.

This helps determine whether:

- the files use a different extension
- the directory depth is different from expected
- the character files are present but not being captured by the current loader

In [ ]:
# Search for text-like files under PROJECT_ROOT/data
candidate_patterns = ["*.txt", "*.csv", "*.pickle", "*.pkl"]

for pattern in candidate_patterns:
    found = list((PROJECT_ROOT / "data").rglob(pattern)) if (PROJECT_ROOT / "data").exists() else []
    print(f"\nPattern {pattern}: {len(found)} files found")
    for p in found[:10]:
        print(p)

## 2D. Fixing the raw character corpus path

The initial file inventory returned zero files because the real directory structure differs from the originally assumed path.

After inspecting the raw-data folders, we identified that the character text files are stored under a deeper nested directory.

This adjustment is necessary before continuing with the inventory and parsing steps.

In [ ]:
# This block is now redundant as it's handled in src/data_loading.py
# We keep it as a reference but updated to use PROJECT_ROOT
from pathlib import Path
from typing import List

def get_project_root() -> Path:
    return PROJECT_ROOT

def get_character_texts_dir() -> Path:
    return PROJECT_ROOT / "data" / "raw" / "movie_characters" / "texts"

def list_character_files(extension: str = "*.txt") -> List[Path]:
    base_dir = get_character_texts_dir()
    return sorted(base_dir.rglob(extension))

## 2. Character file inventory

Before parsing the dataset, we need to understand how the raw character files are organized.

After correcting the raw-data path, we can now inspect the character corpus properly by:

- counting the total number of files
- reviewing a sample of file paths
- understanding how file names and folders encode metadata

This step is important because the path structure will determine how we extract identifiers such as `movie_id` and `character_name`.

In [14]:
character_files = list_character_files()

print("Total character files:", len(character_files))

for path in character_files[:10]:
    print(path)

Total character files: 32114
/Users/jesussalgado/Downloads/movie-character-analysis/data/raw/movie_characters/data/movie_character_texts/movie_character_texts/10 Cloverfield Lane_1179933/Howard_text.txt
/Users/jesussalgado/Downloads/movie-character-analysis/data/raw/movie_characters/data/movie_character_texts/movie_character_texts/10 Cloverfield Lane_1179933/Michelle_text.txt
/Users/jesussalgado/Downloads/movie-character-analysis/data/raw/movie_characters/data/movie_character_texts/movie_character_texts/10 Things I Hate About You_0147800/Bianca Stratford_text.txt
/Users/jesussalgado/Downloads/movie-character-analysis/data/raw/movie_characters/data/movie_character_texts/movie_character_texts/10 Things I Hate About You_0147800/Bogey Lowenstein_text.txt
/Users/jesussalgado/Downloads/movie-character-analysis/data/raw/movie_characters/data/movie_character_texts/movie_character_texts/10 Things I Hate About You_0147800/Cameron James_text.txt
/Users/jesussalgado/Downloads/movie-character-analy

## 3. Raw file content inspection

After confirming how the files are organized, the next step is to inspect their internal content.

The goal of this section is to verify whether the raw character files follow the expected line structure described in the dataset documentation. In particular, we want to confirm that each entry follows a pattern similar to:

`(i)(k) label: text`

where:

- `i` represents a narrative segment
- `k` represents a scene within that segment
- `label` indicates whether the line corresponds to dialogue or descriptive text

## Why this matters

The parser implemented in `src/parsing.py` depends on this structure being consistent.

Before applying the parser to the full corpus, we need to confirm:

- whether the format is stable across files
- whether only `dialog` and `text` appear as labels
- whether there are malformed or unexpected lines
- whether any preprocessing adjustment is needed before large-scale parsing

## Expected output

At the end of this section, we expect to have a clear understanding of the raw text format and any edge cases that may affect parsing.

In [15]:
# Inspect the first few files manually
sample_files = character_files[:3]

for file_path in sample_files:
    print("=" * 100)
    print("FILE:", file_path.name)
    print("MOVIE FOLDER:", file_path.parent.name)
    print("-" * 100)

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        for idx, line in enumerate(f):
            print(repr(line.rstrip("\n")))
            if idx >= 14:   # show first 15 lines
                break

FILE: Howard_text.txt
MOVIE FOLDER: 10 Cloverfield Lane_1179933
----------------------------------------------------------------------------------------------------
"0) 0) text: Michelle's eyes slowly open. She's back on the mattress, thin blanket covering her again. A tray of food sits on the floor next to the bed. The two four and shovel are gone. Howard sits on a folding chair by the door, his forearm i now wrapped with a bandage where she hit him with the woo She peers at him through the dim light. Is he sleeping? props herself up. His voice cuts through the darkness, startling her --"
'0) 1) dialog: You should eat the eggs. You’1ll be dreaming of fresh food before long. Michelle looks at the tray of food, then back to Howard - detachment in his voice, intelligence in his weary eyes. combination that makes him hard to read.'
"0) 2) dialog: In a couple of weeks, everything's going to be from a can. Panic sets in again. Michelle grabs her cell. Holds it up"
"0) 4) dialog: You've got 

## 4. Parser validation on sample files

After inspecting the raw file content, we now validate the parsing logic on a small subset of files.

The goal of this step is to ensure that:

- the parser correctly extracts segment and scene identifiers
- labels (`dialog`, `text`) are captured properly
- text content is preserved without corruption
- malformed lines are detected and handled

## Why this matters

Before applying the parser to the full dataset, we need to confirm that the parsing logic works reliably on real data.

Any issue at this stage would propagate errors across the entire dataset.

## Expected output

We expect to obtain:

- a small DataFrame with correctly parsed records
- a clear view of the resulting schema
- a list (if any) of parsing errors

In [16]:
# Select a small sample of files
sample_files = character_files[:5]

df_sample, df_errors_sample = build_character_lines_dataset(sample_files)

print("Parsed sample shape:", df_sample.shape)
print("Errors in sample:", df_errors_sample.shape)

df_sample.head(10)

Parsed sample shape: (0, 0)
Errors in sample: (1003, 3)


""


## 4A. Unit test for the parsing function

Before applying the parser to full files, we validate the core parsing function on a few manually defined examples.

This helps isolate whether the issue comes from:

- the regular expression itself
- the `parse_line()` implementation
- or a later stage in the file-level parsing pipeline

## Why this matters

The previous sample parsing attempt returned zero valid rows, which indicates that the parsing logic is still not matching the real raw text format.

A direct unit-level validation is the fastest and safest way to diagnose the issue before continuing.

In [17]:
test_lines = [
    "0) 0) text: Michelle's eyes slowly open.",
    "0) 1) dialog: You should eat the eggs.",
    "1) 2) dialog: It took you ten minutes to close your books and come downstairs?",
    "malformed line"
]

for line in test_lines:
    print("INPUT :", repr(line))
    print("OUTPUT:", parse_line(line))
    print("-" * 80)

INPUT : "0) 0) text: Michelle's eyes slowly open."
OUTPUT: None
--------------------------------------------------------------------------------
INPUT : '0) 1) dialog: You should eat the eggs.'
OUTPUT: None
--------------------------------------------------------------------------------
INPUT : '1) 2) dialog: It took you ten minutes to close your books and come downstairs?'
OUTPUT: None
--------------------------------------------------------------------------------
INPUT : 'malformed line'
OUTPUT: None
--------------------------------------------------------------------------------


In [18]:
import re

pattern = re.compile(r"^\s*(\d+)\)\s*(\d+)\)\s*(dialog|text):\s*(.*)$")

test_lines = [
    "0) 0) text: Michelle's eyes slowly open.",
    "0) 1) dialog: You should eat the eggs.",
    "1) 2) dialog: It took you ten minutes to close your books and come downstairs?",
    "malformed line"
]

for line in test_lines:
    match = pattern.match(line)
    print("INPUT :", repr(line))
    print("MATCH :", match.groups() if match else None)
    print("-" * 80)

INPUT : "0) 0) text: Michelle's eyes slowly open."
MATCH : ('0', '0', 'text', "Michelle's eyes slowly open.")
--------------------------------------------------------------------------------
INPUT : '0) 1) dialog: You should eat the eggs.'
MATCH : ('0', '1', 'dialog', 'You should eat the eggs.')
--------------------------------------------------------------------------------
INPUT : '1) 2) dialog: It took you ten minutes to close your books and come downstairs?'
MATCH : ('1', '2', 'dialog', 'It took you ten minutes to close your books and come downstairs?')
--------------------------------------------------------------------------------
INPUT : 'malformed line'
MATCH : None
--------------------------------------------------------------------------------


In [19]:
import importlib
import src.parsing as parsing_module

importlib.reload(parsing_module)

parse_line = parsing_module.parse_line
parse_character_file = parsing_module.parse_character_file
build_character_lines_dataset = parsing_module.build_character_lines_dataset

## 4B. Reloading the parsing module and validating the fix

After identifying that the regular expression works correctly in isolation, we reload the parsing module to ensure the notebook is using the updated version of the parser.

This step is necessary because notebooks can retain older imported versions of local modules even after the source code has been edited.

## Why this matters

The previous parsing attempt returned zero valid rows, but direct regex validation confirmed that the expected line format is correct.

At this point, we need to verify that the updated parser is now being applied consistently through the reusable functions defined in `src/parsing.py`.

In [20]:
import importlib
import src.parsing as parsing_module

importlib.reload(parsing_module)

parse_line = parsing_module.parse_line
parse_character_file = parsing_module.parse_character_file
build_character_lines_dataset = parsing_module.build_character_lines_dataset

# Re-run the unit test
test_lines = [
    "0) 0) text: Michelle's eyes slowly open.",
    "0) 1) dialog: You should eat the eggs.",
    "1) 2) dialog: It took you ten minutes to close your books and come downstairs?",
    "malformed line"
]

for line in test_lines:
    print("INPUT :", repr(line))
    print("OUTPUT:", parse_line(line))
    print("-" * 80)

INPUT : "0) 0) text: Michelle's eyes slowly open."
OUTPUT: {'segment_id': 0, 'scene_id': 0, 'scene_key': '0_0', 'label': 'text', 'text': "Michelle's eyes slowly open.", 'word_count': 4, 'text_length': 28}
--------------------------------------------------------------------------------
INPUT : '0) 1) dialog: You should eat the eggs.'
OUTPUT: {'segment_id': 0, 'scene_id': 1, 'scene_key': '0_1', 'label': 'dialog', 'text': 'You should eat the eggs.', 'word_count': 5, 'text_length': 24}
--------------------------------------------------------------------------------
INPUT : '1) 2) dialog: It took you ten minutes to close your books and come downstairs?'
OUTPUT: {'segment_id': 1, 'scene_id': 2, 'scene_key': '1_2', 'label': 'dialog', 'text': 'It took you ten minutes to close your books and come downstairs?', 'word_count': 12, 'text_length': 64}
--------------------------------------------------------------------------------
INPUT : 'malformed line'
OUTPUT: None
-------------------------------

## 4C. Parsing a small sample of files

Once the updated parser has been validated on manual examples, the next step is to test it on a small subset of real character files.

This allows us to confirm that the full file-level pipeline works correctly before scaling the parsing process to the entire corpus.

## Expected output

At this stage, we expect:

- a non-empty parsed sample dataset
- correctly populated metadata fields such as `movie_id` and `character_name`
- a manageable number of parsing errors, ideally close to zero

In [21]:
sample_files = character_files[:5]

df_sample, df_errors_sample = build_character_lines_dataset(sample_files)

print("Parsed sample shape:", df_sample.shape)
print("Errors in sample:", df_errors_sample.shape)

df_sample.head(10)

Parsed sample shape: (1003, 11)
Errors in sample: (0, 0)


,movie_id,character_name,source_file,segment_id,scene_id,scene_key,label,text,word_count,text_length,line_number
0,10 Cloverfield Lane_1179933,Howard,/Users/jesussalgado/Downloads/movie-character-...,0,0,0_0,text,Michelle's eyes slowly open. She's back on the...,80,417,1
1,10 Cloverfield Lane_1179933,Howard,/Users/jesussalgado/Downloads/movie-character-...,0,1,0_1,dialog,You should eat the eggs. You’1ll be dreaming o...,41,227,2
2,10 Cloverfield Lane_1179933,Howard,/Users/jesussalgado/Downloads/movie-character-...,0,2,0_2,dialog,"In a couple of weeks, everything's going to be...",23,116,3
3,10 Cloverfield Lane_1179933,Howard,/Users/jesussalgado/Downloads/movie-character-...,0,4,0_4,dialog,"You've got no idea what happened, do you? Mich...",15,87,4
4,10 Cloverfield Lane_1179933,Howard,/Users/jesussalgado/Downloads/movie-character-...,0,5,0_5,dialog,There was an attack last night. Some kind of b...,22,127,5
5,10 Cloverfield Lane_1179933,Howard,/Users/jesussalgado/Downloads/movie-character-...,0,7,0_7,dialog,I was rushing back here when I came across you...,18,93,6
6,10 Cloverfield Lane_1179933,Howard,/Users/jesussalgado/Downloads/movie-character-...,0,11,0_11,dialog,I think you're smart enough to know that if I ...,20,99,7
7,10 Cloverfield Lane_1179933,Howard,/Users/jesussalgado/Downloads/movie-character-...,0,13,0_13,dialog,I’m a doctor. I had to take off your jeans to ...,14,60,8
8,10 Cloverfield Lane_1179933,Howard,/Users/jesussalgado/Downloads/movie-character-...,0,15,0_15,dialog,You can’t leave. The entire countryside is bla...,15,98,9
9,10 Cloverfield Lane_1179933,Howard,/Users/jesussalgado/Downloads/movie-character-...,0,17,0_17,dialog,"You go outside, you die. But this place has a ...",25,133,10


## 5. Full corpus parsing

After validating the parser on a small sample of files, the next step is to apply it to the full character corpus.

The goal of this section is to construct the first project-wide analytical dataset, with one row per character text entry across all available movies.

## Why this matters

The previous sample confirmed that the parser works correctly on real files and that the extracted schema is appropriate for downstream analysis.

Scaling the process to the full corpus will allow us to:

- quantify the size of the dataset
- assess parsing stability at scale
- build the intermediate table that will support preprocessing and later analysis stages

## Expected output

At the end of this section, we expect to obtain:

- a full parsed dataset covering the entire corpus
- the total number of extracted records
- the total number of parsing errors
- a first project-wide view of the data structure

In [22]:
df_lines, df_errors = build_character_lines_dataset(character_files)

print("Full parsed dataset shape:", df_lines.shape)
print("Full parsing errors shape:", df_errors.shape)

print("\nColumns:")
print(df_lines.columns.tolist())

df_lines.head()

Full parsed dataset shape: (1462718, 11)
Full parsing errors shape: (0, 0)

Columns:
['movie_id', 'character_name', 'source_file', 'segment_id', 'scene_id', 'scene_key', 'label', 'text', 'word_count', 'text_length', 'line_number']


,movie_id,character_name,source_file,segment_id,scene_id,scene_key,label,text,word_count,text_length,line_number
0,10 Cloverfield Lane_1179933,Howard,/Users/jesussalgado/Downloads/movie-character-...,0,0,0_0,text,Michelle's eyes slowly open. She's back on the...,80,417,1
1,10 Cloverfield Lane_1179933,Howard,/Users/jesussalgado/Downloads/movie-character-...,0,1,0_1,dialog,You should eat the eggs. You’1ll be dreaming o...,41,227,2
2,10 Cloverfield Lane_1179933,Howard,/Users/jesussalgado/Downloads/movie-character-...,0,2,0_2,dialog,"In a couple of weeks, everything's going to be...",23,116,3
3,10 Cloverfield Lane_1179933,Howard,/Users/jesussalgado/Downloads/movie-character-...,0,4,0_4,dialog,"You've got no idea what happened, do you? Mich...",15,87,4
4,10 Cloverfield Lane_1179933,Howard,/Users/jesussalgado/Downloads/movie-character-...,0,5,0_5,dialog,There was an attack last night. Some kind of b...,22,127,5


The full corpus parsing step successfully produced a structured dataset containing 1,462,718 records, with zero parsing errors.

This result indicates that the raw character files follow a highly consistent format, allowing the parsing pipeline to scale reliably across the entire dataset.

The resulting table provides a clean and unified representation of character-level narrative data, including:

- movie identifiers
- character names
- narrative segment and scene approximations
- dialogue and descriptive text labels
- basic text-level features such as word count and text length

At this stage, the project has transitioned from raw semi-structured screenplay data to a fully structured analytical dataset.

This dataset will serve as the foundation for all subsequent analysis, including emotional profiling, interaction graph construction, and character evolution modeling.

----

## 6. Initial data quality checks

Now that the full dataset has been constructed, the next step is to perform an initial round of data quality and exploratory checks.

The goal of this section is to understand the basic properties of the dataset before moving into preprocessing and feature engineering.

## Why this matters

Even though the parsing process completed successfully, it is still necessary to validate:

- the distribution of labels (`dialog` vs `text`)
- the range and behavior of narrative segments and scenes
- the distribution of text lengths
- how data is distributed across movies and characters

These checks help identify potential inconsistencies, biases, or unexpected patterns in the dataset.

## Expected output

At the end of this section, we expect to obtain:

- label distribution
- descriptive statistics for key numerical fields
- an overview of dataset balance across movies and characters

In [23]:
df_lines["label"].value_counts()

label
dialog    1002581
text       460137
Name: count, dtype: int64

In [24]:
df_lines[["segment_id", "scene_id"]].describe()

,segment_id,scene_id
count,1.462718e+06,1.462718e+06
mean,5.516067e+01,4.254606e+01
std,4.936268e+01,1.342690e+02
min,0.000000e+00,0.000000e+00
25%,1.700000e+01,3.000000e+00
50%,4.300000e+01,1.000000e+01
75%,8.000000e+01,2.800000e+01
max,7.350000e+02,3.348000e+03


In [25]:
df_lines.groupby("movie_id").size().sort_values(ascending=False).head(10)

movie_id
Steve Jobs_2080374                             3498
The Curious Case of Benjamin Button_0421715    3200
Hannah and Her Sisters_0091167                 2675
Ed Wood_0109707                                2402
Forrest Gump_0109830                           2379
Life as a House_0264796                        2309
Mary Queen of Scots_2328900                    2222
The Fifth Estate_1837703                       2216
Babel_0449467                                  2098
Little Athens_0417907                          2095
dtype: int64

The initial data quality checks reveal several important characteristics of the dataset.

### Label distribution

The dataset is dominated by dialogue, with over 1 million `dialog` entries compared to approximately 460,000 `text` entries. This indicates that the dataset is primarily composed of spoken interactions, which is particularly useful for tasks such as:

- character interaction analysis
- dialogue-based emotion modeling
- network construction based on co-presence in scenes

At the same time, the presence of descriptive `text` entries provides narrative context, which can be valuable for understanding scene dynamics and character environments.

### Narrative structure (segment and scene)

The `segment_id` and `scene_id` distributions show a wide range of values:

- segments reach values above 700
- scenes can exceed 3000 in some cases

This suggests that the `(segment_id, scene_id)` pair does not map directly to traditional cinematic scenes, but rather to a finer-grained structural segmentation of the screenplay.

Despite this, the identifiers still provide a consistent and ordered approximation of narrative progression, which makes them suitable for:

- defining scene-level co-occurrence between characters
- analyzing temporal evolution of characters across the narrative

### Data distribution across movies

There is noticeable variability in the number of lines per movie. Some films contain more than 3,000 entries, while others have significantly fewer.

This imbalance is expected in screenplay datasets and reflects differences in:

- script length
- narrative style
- number of characters and interactions

However, this also implies that downstream analyses should account for this variability, especially when comparing movies or aggregating results globally.

### Key implications

From these checks, we can conclude that:

- the dataset is structurally consistent and ready for further processing
- dialogue dominates the dataset, making it well-suited for interaction and emotion analysis
- `(segment_id, scene_id)` provides a reliable operational definition of narrative context
- there is natural variability across movies that should be considered in later stages

These insights confirm that the dataset is suitable for building character-level analytical features in the next phase.

## 7. Saving the interim dataset

After validating the parsing process and performing initial data quality checks, the dataset is now ready to be stored as an intermediate artifact.

This dataset will serve as the input for the next phase of the project, where preprocessing, feature engineering, and aggregation will be applied.

Saving the dataset at this stage ensures:

- reproducibility of results
- separation between raw data and processed data
- faster iteration in downstream notebooks

In [ ]:
interim_dir = PROJECT_ROOT / "data" / "interim"
interim_dir.mkdir(parents=True, exist_ok=True)

df_lines.to_parquet(interim_dir / "character_lines.parquet", index=False)

print("Dataset saved to data/interim/character_lines.parquet")

## Conclusion

In this notebook, we successfully transformed the raw character-level screenplay files into a structured analytical dataset.

Key achievements include:

- validating the real structure of the dataset
- designing and implementing a robust parsing pipeline
- extracting over 1.4 million character-level records
- confirming data consistency with zero parsing errors
- performing initial data quality checks

This dataset now provides a reliable foundation for the next phase of the project, where we will focus on preprocessing and feature engineering to support character-level analysis.

The combination of structured narrative identifiers and text-level features enables future tasks such as:

- emotion analysis
- interaction graph construction
- character role detection